# Logit Lens Per-Layer Last-Token Vectors for T5

This notebook runs a prompt through flan-t5-base, captures `SelfAttention.o` activations at each layer,
applies the logit lens (unembedding matrix), and builds an ordered list of logit vectors —
one per attention layer in forward-pass order (encoder block 0 → encoder block N → decoder blocks).

h## Cell 1 — Imports & setup

In [ ]:
import sys
import os

# Make sure the neuralsignal package is importable when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch

from neuralsignal.core.modules.model_instrumentation import load_model, generate_from_batch
from neuralsignal.core.modules.feature_sets.feature_set_logit_lens import FeatureSetLogitLens

MODEL_NAME = "google/flan-t5-base"
HF_TOKEN   = ""          # set if needed for gated models
DEVICE     = "cpu"       # change to "cuda:0" if a GPU is available
TOP_K      = 5           # number of top tokens to show per layer

print(f"torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Cell 2 — Load model

In [ ]:
tokenizer, model = load_model({
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "quantization": "no_quantization",
})

print(f"Model loaded : {model.name_or_path}")
print(f"Model device : {model.device}")
print(f"lm_head shape: {model.lm_head.weight.shape}")

## Cell 3 — Configure instrumentation

Key settings:
- `layer_names_to_include: ["SelfAttention.o"]` — only capture attention output projections
- `zone_size_by_layer: {"default": 1}` — zone size of 1 disables avg_pool1d, preserving the full hidden-dim vector
  *(Note: the `zone_size` key alone is not sufficient due to a known quirk in `Collector.__init__`;
  `zone_size_by_layer` with a `"default"` entry must be used instead.)*
- `instrument_FF: False`, `instrument_embedding: False` — reduces hook count to attention layers only

In [ ]:
instrumentation_cfg = {
    "instrument_encoder"  : True,
    "instrument_decoder"  : True,
    "instrument_FF"       : False,
    "instrument_attention": True,
    "instrument_embedding": False,
    "collector_config": {
        "mode"          : "additive",
        "data_to_save"  : ["outputs", "layer_info", "topology"],
        # zone_size_by_layer with default=1 preserves full hidden-dim vectors
        "zone_size_by_layer"      : {"default": 1},
        "layer_names_to_include"  : ["SelfAttention.o"],
    },
}

print("Instrumentation config ready.")

## Cell 4 — Run prompt and collect activations

In [ ]:
prompts = ["translate English to French: The cat is on the mat"]

gis = generate_from_batch(
    prompts, model, tokenizer,
    instrumentation_cfg=instrumentation_cfg
)
gi = gis[0]

print(f"Input  : {gi.data['input']}")
print(f"Output : {gi.data['decoded_output']}")
print(f"Layers captured: {len(gi.data['layer_order'])}")
print(f"Layer names sample: {gi.data['layer_names'][:4]}")

## Cell 5 — Set up logit lens

Pass `unembed_layer=model.lm_head` to reuse the already-loaded weight matrix
instead of loading a second copy of T5.

In [ ]:
logit_lens = FeatureSetLogitLens({
    "model_name"      : MODEL_NAME,   # only used if unembed_layer is absent
    "hf_token"        : HF_TOKEN,
    "layers_to_process": ["SelfAttention.o"],
    "unembed_layer"   : model.lm_head,
    "output_format"   : "name_and_value_columns",
    "dev_map"         : DEVICE,
})

print(f"Logit lens unembed type : {type(logit_lens.unembed)}")
print(f"Vocab size              : {logit_lens.unembed.weight.shape[0]}")

## Cell 6 — Build the ordered list of per-layer logit vectors

For each `SelfAttention.o` layer (in forward-pass order):
1. Retrieve the activation tensor — shape `(seq_len, hidden_dim)` (batch dim was stripped by the Collector)
2. Take the **last-token** slice: `tensor[-1:, :]` → `(1, hidden_dim)`
3. Apply the unembedding matrix → `(1, vocab_size)`

**Note on decoder layers:** the Collector runs in `additive` mode, so decoder tensors
accumulate across all autoregressive generation steps. The last-token slice therefore
represents the *summed* activation at position -1 across all decoder passes,
not just the final generation step. Encoder tensors are unaffected (single pass).

In [ ]:
layer_id_to_name = gi.data["layer_id_to_name"]
layer_order      = gi.data["layer_order"]
outputs          = gi.data["outputs"]

logit_vectors = []   # ordered list — one entry per SelfAttention.o layer

for layer_id in layer_order:
    layer_name = layer_id_to_name[layer_id]
    if "SelfAttention.o" not in layer_name:
        continue

    # shape: (seq_len, hidden_dim)  — batch dim already removed by Collector
    tensor = outputs[layer_id]
    last_token = tensor[-1:, :]          # (1, hidden_dim)

    # Move to the correct device before unembedding
    last_token = last_token.to(DEVICE)

    with torch.no_grad():
        logit_vec = logit_lens.unembed.forward(last_token)   # (1, vocab_size)

    logit_vectors.append({
        "layer_name"  : layer_name,
        "logit_vector": logit_vec[0],    # (vocab_size,) tensor
    })

print(f"Total SelfAttention.o layers captured: {len(logit_vectors)}")
print(f"Expected for flan-t5-base: 24 (12 encoder + 12 decoder)")
if logit_vectors:
    print(f"logit_vector shape: {logit_vectors[0]['logit_vector'].shape}")
    print(f"Expected vocab size: 32128")

## Cell 7 — Inspect results: top-k tokens per layer

In [ ]:
print(f"{'Layer':<60}  {'Top-' + str(TOP_K) + ' tokens'}")
print("-" * 100)

for entry in logit_vectors:
    vec   = entry["logit_vector"]
    name  = entry["layer_name"]

    topk_vals, topk_ids = torch.topk(vec, TOP_K)
    top_tokens = tokenizer.convert_ids_to_tokens(topk_ids.tolist())

    token_str = ", ".join(
        f"{tok}({val:.2f})" for tok, val in zip(top_tokens, topk_vals.tolist())
    )
    print(f"{name:<60}  {token_str}")

## Cell 8 — Quick sanity checks

In [ ]:
VOCAB_SIZE = 32128

assert len(logit_vectors) == 24, (
    f"Expected 24 attention layers for flan-t5-base, got {len(logit_vectors)}"
)
for entry in logit_vectors:
    shape = entry["logit_vector"].shape
    assert shape == (VOCAB_SIZE,), (
        f"Expected logit vector shape ({VOCAB_SIZE},), got {shape} "
        f"for layer {entry['layer_name']}"
    )

encoder_layers = [e for e in logit_vectors if e["layer_name"].startswith("encoder.")]
decoder_layers = [e for e in logit_vectors if e["layer_name"].startswith("decoder.")]

print(f"Encoder SelfAttention.o layers: {len(encoder_layers)}  (expected 12)")
print(f"Decoder SelfAttention.o layers: {len(decoder_layers)}  (expected 12)")
print()
print("All assertions passed.")
print(f"  logit_vectors[0]  = first encoder attention layer : {logit_vectors[0]['layer_name']}")
print(f"  logit_vectors[-1] = last  decoder attention layer : {logit_vectors[-1]['layer_name']}")